In [ ]:
import pandas as pd
import numpy as np
import os
import glob
from tqdm import tqdm
import xgboost as xgb
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import skew, kurtosis
import warnings

# 忽略烦人的警告
warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', None)

In [ ]:
# -----------------------------------------------------------------------------
# 1. 配置模块 (Configuration)
# -----------------------------------------------------------------------------
class Config:
    # 路径配置
    PATH_DAILY = './data/日行情/'
    PATH_HF = './data/高频数据/'
    PATH_WEIGHT = './data/指数权重/'
    PATH_IND = './data/股票行业分类/'
    PATH_CAP = './data/股票市值/'
    POOL_FILE = 'index_wt_000905.parquet'
    
    # 时间配置 (聚焦数据质量最好的区间)
    START_DATE = '2016-01-01'
    END_DATE = '2018-12-31'
    
    # 训练配置
    TRAIN_WINDOW = 12 * 20  # 滚动训练窗口: 12个月 (约240交易日)
    PREDICT_WINDOW = 20     # 每次预测: 1个月
    
    # 成本配置
    COST = 0.0015  # 双边万分之15

In [ ]:
# -----------------------------------------------------------------------------
# 2. 数据加载模块 (Data Loader) - 保持稳健性
# -----------------------------------------------------------------------------
class DataLoader:
    def __init__(self, cfg):
        self.cfg = cfg

    def _format_code(self, s):
        return s.astype(str).str.zfill(6)

    def load_all(self):
        print(">>> [Step 1] Loading Data...")
        
        # 1. 读取股票池 (Universe)
        df_univ = pd.read_parquet(os.path.join(self.cfg.PATH_WEIGHT, self.cfg.POOL_FILE))
        df_univ = df_univ.rename(columns={'symbol': 'SecuCode', 'date': 'trade_date'})
        df_univ['SecuCode'] = self._format_code(df_univ['SecuCode'])
        df_univ['trade_date'] = pd.to_datetime(df_univ['trade_date'])
        
        # 2. 读取日行情
        all_daily = []
        for year in range(2016, 2019): # 2016-2018
            fpath = os.path.join(self.cfg.PATH_DAILY, f"daily_quote_{year}.csv")
            if os.path.exists(fpath):
                all_daily.append(pd.read_csv(fpath))
        df_daily = pd.concat(all_daily, ignore_index=True)
        
        # 格式化
        df_daily = df_daily.rename(columns={'TradingDay': 'trade_date'})
        df_daily['trade_date'] = pd.to_datetime(df_daily['trade_date'])
        df_daily['SecuCode'] = self._format_code(df_daily['SecuCode'])
        
        # 停牌过滤
        df_daily = df_daily[df_daily['PTFlag'] == 0]
        
        # 对齐股票池 (Inner Join)
        df = pd.merge(df_daily, df_univ[['trade_date', 'SecuCode']], on=['trade_date', 'SecuCode'], how='inner')
        
        # 3. 计算复权价和收益率
        df = df.sort_values(['SecuCode', 'trade_date'])
        # T+1 收益率 (Label) -> 必须 shift(-1)
        df['next_ret'] = df.groupby('SecuCode')['ret'].shift(-1)
        # 复权收盘价
        df['adj_factor'] = (1 + df['ret']).groupby(df['SecuCode']).cumprod()
        df['adj_close'] = df['ClosePrice'] * df['adj_factor'] / df['adj_factor'].groupby(df['SecuCode']).transform('first')
        
        # 4. 合并行业与市值 (用于中性化/分析)
        # (此处为简化代码，假设已完成上个版本的 merge_asof 逻辑，若未完成请参考上一版代码)
        # 暂时只做因子计算演示
        
        return df

In [ ]:
# -----------------------------------------------------------------------------
# 3. 因子工厂 (Factor Engine) - 深度与微观结构
# -----------------------------------------------------------------------------
class FactorEngine:
    def __init__(self, cfg):
        self.cfg = cfg
        
    def calc_low_freq(self, df):
        print(">>> [Step 2.1] Calculating Low-Freq Factors (Depth Upgrade)...")
        # 基础数据
        P = df['adj_close']
        V = df['TradingVolumes']
        R = df['ret']
        
        # 1. F_MOM20: 动量
        df['F_MOM20'] = R.groupby(df['SecuCode']).rolling(20).sum().reset_index(0, drop=True)
        
        # 2. F_SKEW20: 收益率偏度 (高阶矩，捕捉尾部风险)
        # 偏度 > 0 通常意味着暴涨概率大，但也可能意味着彩票型股票
        df['F_SKEW20'] = R.groupby(df['SecuCode']).rolling(20).skew().reset_index(0, drop=True)
        
        # 3. F_ILLIQ: Amihud 非流动性因子 (Abs(Ret) / Volume)
        # 衡量单位成交量带来的价格冲击
        df['abs_ret_vol'] = R.abs() / (V + 1e-9) # 防止除零
        df['F_ILLIQ'] = df['abs_ret_vol'].groupby(df['SecuCode']).rolling(20).mean().reset_index(0, drop=True)
        
        # 4. F_VOL_STABILITY: 量的稳定性 (Vol Std / Vol Mean)
        vol_mean = V.groupby(df['SecuCode']).rolling(20).mean().reset_index(0, drop=True)
        vol_std = V.groupby(df['SecuCode']).rolling(20).std().reset_index(0, drop=True)
        df['F_VOL_STAB'] = -1 * (vol_std / (vol_mean + 1e-9)) # 越稳越好
        
        return df

    def calc_high_freq(self, df_daily):
        print(">>> [Step 2.2] Calculating High-Freq Factors (Microstructure)...")
        stock_list = df_daily['SecuCode'].unique()
        hf_res = []
        
        # 为了演示，只跑前10只股票。实盘请去掉 [:10]
        # 提速技巧：实际应使用 joblib 并行处理
        for code in tqdm(stock_list[:20], desc="HF Processing"): 
            fpath = os.path.join(self.cfg.PATH_HF, f"{code}.csv")
            if not os.path.exists(fpath): continue
            
            try:
                df_min = pd.read_csv(fpath)
                # 假设格式: time, close, volume
                # 简单解析日期
                if 'time' in df_min.columns:
                    df_min['date'] = pd.to_datetime(df_min['time']).dt.date
                else: continue
                
                # --- 高频因子逻辑 ---
                def get_daily_hf(g):
                    if len(g) < 30: return pd.Series([np.nan]*3)
                    
                    # 因子5: F_HF_CPV (加权量价相关性)
                    # 逻辑：下午的成交量含金量更高，给予更高权重 (此处简化为普通 corr)
                    cpv = g['close'].corr(g['volume'])
                    
                    # 因子6: F_HF_DownVol (下行波动率)
                    # 逻辑：只看跌的那几分钟的波动率，衡量恐慌抛售压力
                    ret_min = g['close'].pct_change().fillna(0)
                    down_vol = ret_min[ret_min < 0].std()
                    
                    # 因子7: F_HF_SmartQ (聪明钱因子/量占比)
                    # 逻辑：大单成交带来的涨幅。这里用 (大成交量分钟的涨幅累计) 模拟
                    # 简化版：成交量最大的前20%分钟的平均涨幅
                    top_vol = g['volume'].quantile(0.8)
                    smart_ret = ret_min[g['volume'] >= top_vol].mean()
                    
                    return pd.Series([cpv, down_vol, smart_ret], 
                                     index=['F_HF_CPV', 'F_HF_DownVol', 'F_HF_SmartQ'])

                daily_stats = df_min.groupby('date').apply(get_daily_hf)
                daily_stats['SecuCode'] = code
                hf_res.append(daily_stats)
                
            except: continue
            
        if not hf_res: return df_daily
        
        df_hf = pd.concat(hf_res).reset_index()
        df_hf = df_hf.rename(columns={'date': 'trade_date'})
        df_hf['trade_date'] = pd.to_datetime(df_hf['trade_date'])
        df_hf['SecuCode'] = df_hf['SecuCode'].astype(str).str.zfill(6)
        
        # 滚动平滑 (降低噪音)
        cols = ['F_HF_CPV', 'F_HF_DownVol', 'F_HF_SmartQ']
        df_hf = df_hf.sort_values(['SecuCode', 'trade_date'])
        for c in cols:
            df_hf[c] = df_hf.groupby('SecuCode')[c].rolling(10).mean().reset_index(0, drop=True)
            
        # Merge
        df_out = pd.merge(df_daily, df_hf, on=['trade_date', 'SecuCode'], how='left')
        return df_out

    def preprocess(self, df, factors):
        print(">>> [Step 2.3] Preprocessing (Winsorize + Standardize)...")
        # 简单清洗
        df = df.replace([np.inf, -np.inf], np.nan)
        
        for f in factors:
            # 分组去极值 (Cross-sectional Winsorization)
            def _win_std(x):
                # MAD 去极值
                med = x.median()
                mad = (x - med).abs().median()
                upper = med + 3 * 1.4826 * mad
                lower = med - 3 * 1.4826 * mad
                x = x.clip(lower, upper)
                # Z-Score 标准化
                return (x - x.mean()) / (x.std() + 1e-9)
            
            df[f] = df.groupby('trade_date')[f].transform(_win_std)
            df[f] = df[f].fillna(0) # 缺失值填0
        return df



In [ ]:
# -----------------------------------------------------------------------------
# 4. 因子分析器 (Alpha Analyzer) - 简历级分析
# -----------------------------------------------------------------------------
class AlphaAnalyzer:
    @staticmethod
    def check_correlation(df, factors):
        print("\n--- Factor Correlation Check ---")
        corr = df[factors].corr()
        plt.figure(figsize=(10, 8))
        sns.heatmap(corr, annot=True, cmap='coolwarm', fmt=".2f")
        plt.title("Factor Correlation Matrix")
        plt.show()
        
    @staticmethod
    def calc_ic_ir(df, factors, label='next_ret'):
        print("\n--- IC / IR Analysis ---")
        ic_data = []
        for date, group in df.groupby('trade_date'):
            daily_ic = {}
            for f in factors:
                # Rank IC (Spearman)
                ic = group[f].corr(group[label], method='spearman')
                daily_ic[f] = ic
            ic_data.append(daily_ic)
            
        df_ic = pd.DataFrame(ic_data, index=df['trade_date'].unique()).sort_index()
        
        # 打印结果
        summary = pd.DataFrame()
        summary['IC Mean'] = df_ic.mean()
        summary['IC Std'] = df_ic.std()
        summary['IR'] = summary['IC Mean'] / summary['IC Std']
        summary['IC > 0'] = (df_ic > 0).sum() / len(df_ic)
        
        print(summary)
        
        # 画 IC 累积图
        plt.figure(figsize=(12, 5))
        df_ic.cumsum().plot()
        plt.title("Cumulative Rank IC")
        plt.axhline(0, color='black', linestyle='--')
        plt.show()
        return summary

In [ ]:
# -----------------------------------------------------------------------------
# 5. 滚动训练模型 (Rolling XGBoost) - 严谨回测
# -----------------------------------------------------------------------------
class RollingModel:
    def __init__(self, cfg):
        self.cfg = cfg
        
    def run(self, df, factors):
        print(">>> [Step 3] Running Rolling Window Training...")
        
        dates = sorted(df['trade_date'].unique())
        train_window = self.cfg.TRAIN_WINDOW # e.g. 240 days
        predict_window = self.cfg.PREDICT_WINDOW # e.g. 20 days
        
        predictions = []
        
        # 滚动逻辑
        # Start from: first date + train window
        # Step: predict_window
        for i in tqdm(range(train_window, len(dates), predict_window)):
            # 1. 切分时间
            train_start = dates[i - train_window]
            train_end = dates[i - 1] # 不包含预测日
            test_start = dates[i]
            test_end = dates[min(i + predict_window - 1, len(dates)-1)]
            
            if test_start > dates[-1]: break
            
            # 2. 准备数据
            mask_train = (df['trade_date'] >= train_start) & (df['trade_date'] <= train_end)
            mask_test = (df['trade_date'] >= test_start) & (df['trade_date'] <= test_end)
            
            train_data = df[mask_train].dropna(subset=factors + ['next_ret'])
            test_data = df[mask_test].dropna(subset=factors)
            
            if len(test_data) == 0: continue
            
            X_train = train_data[factors]
            y_train = train_data['next_ret']
            X_test = test_data[factors]
            
            # 3. 训练模型 (XGBoost Ranker or Regressor)
            # 为了速度用 Regressor，实盘常用 Ranker
            model = xgb.XGBRegressor(
                max_depth=3,
                learning_rate=0.1,
                n_estimators=50,
                n_jobs=4,
                objective='reg:squarederror'
            )
            model.fit(X_train, y_train)
            
            # 4. 预测
            pred_score = model.predict(X_test)
            
            res = test_data[['trade_date', 'SecuCode', 'next_ret']].copy()
            res['pred_score'] = pred_score
            predictions.append(res)
            
        df_res = pd.concat(predictions)
        return df_res

In [ ]:
# -----------------------------------------------------------------------------
# 6. 回测与绩效分析 (Performance) - 实战指标
# -----------------------------------------------------------------------------
class Backtester:
    def __init__(self, cfg):
        self.cfg = cfg
        
    def run_layer_test(self, df_res):
        print(">>> [Step 4] Running Backtest (Turnover & Drawdown)...")
        
        # 1. 分层
        df_res['group'] = df_res.groupby('trade_date')['pred_score'].transform(
            lambda x: pd.qcut(x, 5, labels=False, duplicates='drop')
        )
        
        # 2. 计算各组收益 (费前)
        group_ret = df_res.groupby(['trade_date', 'group'])['next_ret'].mean().unstack()
        long_ret = group_ret[4] # Top 20%
        bench_ret = df_res.groupby('trade_date')['next_ret'].mean() # Market
        active_ret = long_ret - bench_ret # 超额
        
        # 3. 计算换手率 (Turnover)
        # 逻辑：Top组每一期持仓权重的变化
        top_stocks = df_res[df_res['group'] == 4]
        # Pivot table: index=date, col=stock, val=1 (equally weighted)
        pos = top_stocks.pivot(index='trade_date', columns='SecuCode', values='pred_score').notna().astype(int)
        # 归一化权重 (1/N)
        pos = pos.div(pos.sum(axis=1), axis=0).fillna(0)
        # 计算权重变化 delta
        turnover = pos.diff().abs().sum(axis=1) / 2 # 单边换手
        avg_turnover = turnover.mean()
        
        # 4. 费后收益 (Net Return)
        # Cost = Turnover * Transaction_Cost
        t_cost = turnover * self.cfg.COST
        net_active_ret = active_ret - t_cost
        
        # 5. 绩效指标
        cum_ret = (1 + net_active_ret).cumprod()
        ann_ret = net_active_ret.mean() * 252
        ann_vol = net_active_ret.std() * np.sqrt(252)
        sharpe = ann_ret / ann_vol
        
        # 最大回撤 (MDD)
        roll_max = cum_ret.cummax()
        drawdown = cum_ret / roll_max - 1
        max_dd = drawdown.min()
        calmar = ann_ret / abs(max_dd)
        
        # 6. 打印报告
        print("\n========== Backtest Report (Hedge Fund Standard) ==========")
        print(f"Annualized Excess Return (Net): {ann_ret:.2%}")
        print(f"Information Ratio / Sharpe:     {sharpe:.2f}")
        print(f"Max Drawdown:                   {max_dd:.2%}")
        print(f"Calmar Ratio:                   {calmar:.2f}")
        print(f"Avg Daily Turnover (One-side):  {avg_turnover:.2%}")
        print("==========================================================")
        
        # 绘图
        plt.figure(figsize=(12, 8))
        
        plt.subplot(2, 1, 1)
        plt.plot((1+long_ret).cumprod(), label='Top Group (Gross)')
        plt.plot((1+bench_ret).cumprod(), label='Benchmark')
        plt.plot(cum_ret, label='Active Net (After Cost)', color='red', linewidth=2)
        plt.title("Net Value Curve")
        plt.legend()
        
        plt.subplot(2, 1, 2)
        plt.fill_between(drawdown.index, drawdown.values, color='green', alpha=0.3)
        plt.title("Drawdown")
        
        plt.tight_layout()
        plt.show()

In [ ]:
# -----------------------------------------------------------------------------
# 主程序入口
# -----------------------------------------------------------------------------
if __name__ == "__main__":
    cfg = Config()
    
    # 1. 加载数据
    loader = DataLoader(cfg)
    df = loader.load_all()
    
    # 2. 计算因子
    eng = FactorEngine(cfg)
    df = eng.calc_low_freq(df)
    df = eng.calc_high_freq(df) # 注意：这里只跑了部分股票用于演示
    
    # 3. 因子列表
    factors = ['F_MOM20', 'F_SKEW20', 'F_ILLIQ', 'F_VOL_STAB'] 
    if 'F_HF_CPV' in df.columns:
        factors.extend(['F_HF_CPV', 'F_HF_DownVol', 'F_HF_SmartQ'])
        
    # 4. 清洗
    df = eng.preprocess(df, factors)
    
    # 5. 分析 (IC/IR)
    analyzer = AlphaAnalyzer()
    analyzer.check_correlation(df, factors)
    analyzer.calc_ic_ir(df, factors)
    
    # 6. 滚动训练
    trainer = RollingModel(cfg)
    df_res = trainer.run(df, factors)
    
    # 7. 回测
    bt = Backtester(cfg)
    bt.run_layer_test(df_res)